In [0]:
import os
import glob
import json
import urllib.request

from datetime import date
from common_utils.logging import get_logger
from common_utils.ingestor import setup_catalog, setup_schema, setup_volume, setup_directory, read_and_write_raw

# ============================================================
# 0. LOGGER
# ============================================================
logger = get_logger("dimension-ingestion")


# ============================================================
# 1. CONFIG PATH
# ============================================================
dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

# ============================================================
# 2. READ CONFIG
# ============================================================
with open(config_path, "r") as f:
    config = json.load(f)

# ============================================================
# 3. CONFIG SECTIONS
# ============================================================
gdrive_file_ids = config["source"]["gdrive_file_ids"]

catalog = config["target"]["catalog"]
schema = config["target"]["schema"]
volume = config["target"]["volume"]

gdrive_base_url = config["source"]["gdrive_base_url"]
base_path = config["target"]["base_path"]
destination_folder = config["target"]["destination_folder"]

# ============================================================
# 4. RUN DATE
# ============================================================
run_date = date.today().isoformat()
logger.info("Load date is %s", run_date)
logger.info("Reading data from %s", gdrive_base_url)

# ============================================================
# 5. OUTPUT PATH
# ============================================================
output_path = f"{base_path}/load_date={run_date}/{destination_folder}"
logger.warning("Data will be stored in the path %s", output_path)

# ============================================================
# 6. CREATING A CATALOG, SCHEMA AND VOLUME IF DOES NOT EXIST
# ============================================================

logger.info("creating catalog if does not exist")
setup_catalog(spark, catalog)

logger.info("creating schema if does not exist")
setup_schema(spark, catalog, schema)

logger.info("creating volume if does not exist")
setup_volume(spark, catalog, schema, volume)

logger.info("Creating a directory in output_path %s", output_path)
setup_directory(dbutils,output_path)

# ===============================================================
# 7. READ DATA FROM GOOGLE DRIVE AND WRITING TO DATABRICKS VOLUME
# ===============================================================
read_and_write_raw(urllib, gdrive_base_url, output_path, gdrive_file_ids)
logger.info("Data succesfully written")
